In [1]:
!pip install pandas nltk scikit-learn sacrebleu


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import json
import re
import ast
from typing import List, Dict, Optional, Tuple
from collections import Counter
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score
import numpy as np
import os

In [3]:
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("✓ Importazioni completate")

✓ Importazioni completate


In [4]:
# Configurazione
MODEL_NAME = "qwen3:4b-instruct"
MODEL_NAME_CLEAN = MODEL_NAME.replace(':', '_')
BASE_PATH = f"../Output Performance Degradation Analysis/{MODEL_NAME_CLEAN}_six_task"

# Task configurati - AGGIORNATO PER 6 TASK
TASK_NAMES = {
    1: "json_only",
    2: "json_translation", 
    3: "json_translation_sentiment",
    4: "json_translation_sentiment_emotion",
    5: "json_translation_sentiment_emotion_topic",
    6: "json_translation_sentiment_emotion_topic_ner"
}

# Campi richiesti per ogni task - AGGIORNATO PER 6 TASK
REQUIRED_FIELDS = {
    1: ["review"],
    2: ["review", "translate"],
    3: ["review", "translate", "sentiment"],
    4: ["review", "translate", "sentiment", "emotion"],
    5: ["review", "translate", "sentiment", "emotion", "topic"],
    6: ["review", "translate", "sentiment", "emotion", "topic", "entities"]
}

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - Percorso base: {BASE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")

✓ Configurazione completata:
  - Modello: qwen3:4b-instruct
  - Percorso base: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task
  - Task configurati: 6


In [5]:
def is_valid_json_format(json_string: str, required_fields: List[str]) -> bool:
    """
    Verifica se il JSON è formalmente valido e ha tutti i campi richiesti.
    
    Args:
        json_string (str): Il JSON da validare
        required_fields (List[str]): I campi richiesti
        
    Returns:
        bool: True se valido, False altrimenti
    """
    if pd.isna(json_string) or json_string is None:
        return False
    
    try:
        # Parse del JSON
        json_obj = json.loads(json_string)
        
        # Verifica che sia un dizionario
        if not isinstance(json_obj, dict):
            return False
        
        # Verifica che abbia tutti i campi richiesti
        json_fields = set(json_obj.keys())
        required_fields_set = set(required_fields)
        
        # Deve avere tutti i campi richiesti e nessun campo extra
        if json_fields != required_fields_set:
            return False
            
        return True
        
    except (json.JSONDecodeError, TypeError):
        return False

In [6]:
def calculate_bleu_score(reference: str, candidate: str) -> float:
    """
    Calcola il BLEU score tra reference e candidate.
    
    Args:
        reference (str): Testo di riferimento
        candidate (str): Testo candidato
        
    Returns:
        float: BLEU score (0-100)
    """
    if pd.isna(reference) or pd.isna(candidate) or reference == "" or candidate == "":
        return 0.0
    
    try:
        # Tokenizza i testi
        ref_tokens = nltk.word_tokenize(reference.lower())
        cand_tokens = nltk.word_tokenize(candidate.lower())
        
        # Calcola BLEU score con smoothing
        smoothing = SmoothingFunction()
        score = sentence_bleu([ref_tokens], cand_tokens, 
                            smoothing_function=smoothing.method1)
        
        return score * 100  # Converti in percentuale
        
    except Exception:
        return 0.0

In [7]:
def exact_match_text(reference: str, candidate: str) -> bool:
    """
    Verifica exact match tra due testi (case-insensitive, spazi normalizzati).
    
    Args:
        reference (str): Testo di riferimento
        candidate (str): Testo candidato
        
    Returns:
        bool: True se match esatto, False altrimenti
    """
    if pd.isna(reference) or pd.isna(candidate):
        return False
    
    # Normalizza gli spazi e converti in lowercase
    ref_normalized = ' '.join(str(reference).lower().strip().split())
    cand_normalized = ' '.join(str(candidate).lower().strip().split())
    
    return ref_normalized == cand_normalized

In [8]:
def parse_entities(entities_str: str) -> List[Dict]:
    """
    Parsifica la stringa delle entità in una lista di dizionari.
    
    Args:
        entities_str (str): Stringa delle entità
        
    Returns:
        List[Dict]: Lista delle entità parsificate
    """
    if pd.isna(entities_str) or entities_str == "":
        return []
    
    try:
        # Prova a parsificare come JSON
        if entities_str.startswith('['):
            return json.loads(entities_str)
        
        # Prova a parsificare come literal Python
        return ast.literal_eval(entities_str)
        
    except (json.JSONDecodeError, ValueError, SyntaxError):
        return []

In [9]:
def calculate_ner_f1(reference_entities: str, candidate_entities: List[Dict]) -> float:
    """
    Calcola F1 score per NER.
    
    Args:
        reference_entities (str): Entità di riferimento
        candidate_entities (List[Dict]): Entità candidate
        
    Returns:
        float: F1 score (0-100)
    """
    # Parsifica le entità di riferimento
    ref_entities = parse_entities(reference_entities)
    
    if not ref_entities and not candidate_entities:
        return 100.0  # Entrambi vuoti = match perfetto
    
    if not ref_entities or not candidate_entities:
        return 0.0  # Uno vuoto, uno no = no match
    
    # Estrai set di (label, value) per confronto
    ref_set = set()
    for entity in ref_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            ref_set.add((entity['label'], entity['value'].lower().strip()))
    
    cand_set = set()
    for entity in candidate_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            cand_set.add((entity['label'], entity['value'].lower().strip()))
    
    if not ref_set and not cand_set:
        return 100.0
    
    if not ref_set or not cand_set:
        return 0.0
    
    # Calcola precisione, recall, F1
    intersection = ref_set.intersection(cand_set)
    
    if len(intersection) == 0:
        return 0.0
    
    precision = len(intersection) / len(cand_set)
    recall = len(intersection) / len(ref_set)
    
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1 * 100  # Converti in percentuale

print("✓ Tutte le funzioni di valutazione definite")

✓ Tutte le funzioni di valutazione definite


In [14]:
def evaluate_task_file(task_number: int) -> pd.DataFrame:
    """
    Valuta un singolo file di task e calcola tutte le metriche.
    
    Args:
        task_number (int): Numero del task (1-6)
        
    Returns:
        pd.DataFrame: DataFrame con tutte le metriche calcolate
    """
    task_name = TASK_NAMES[task_number]
    required_fields = REQUIRED_FIELDS[task_number]
    
    # Percorso file
    input_file = f"{BASE_PATH}_cleaned/sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}_cleaned.csv"
    
    print(f"Valutando Task {task_number} ({task_name})...")
    
    # Verifica esistenza file
    if not os.path.exists(input_file):
        print(f"❌ File non trovato: {input_file}")
        return pd.DataFrame()
    
    # Carica il dataframe
    df = pd.read_csv(input_file)
    print(f"✓ File caricato: {len(df)} righe")
    
    # Inizializza TUTTE le colonne metriche
    df['json_valid'] = 0
    df['bleu_translate'] = 0.0
    df['em_sentiment'] = 0
    df['em_emotion'] = 0      # AGGIUNTA
    df['em_topic'] = 0        # AGGIUNTA
    df['f1_ner'] = 0.0
    
    # Calcola metriche per ogni riga
    for idx, row in df.iterrows():
        json_output = row.get('json_output', None)
        
        # 1. Validazione JSON
        df.at[idx, 'json_valid'] = 1 if is_valid_json_format(json_output, required_fields) else 0
        
        # Se JSON non è valido, salta le altre metriche
        if df.at[idx, 'json_valid'] == 0:
            continue
        
        try:
            json_obj = json.loads(json_output)
            
            # 2. BLEU su translate (se richiesto)
            if 'translate' in required_fields:
                ref_translate = row.get('translate', '')
                cand_translate = json_obj.get('translate', '')
                df.at[idx, 'bleu_translate'] = calculate_bleu_score(ref_translate, cand_translate)
            
            # 3. EM su sentiment (se richiesto)
            if 'sentiment' in required_fields:
                ref_sentiment = row.get('sentiment', '')
                cand_sentiment = json_obj.get('sentiment', '')
                df.at[idx, 'em_sentiment'] = 1 if exact_match_text(ref_sentiment, cand_sentiment) else 0
            
            # 4. EM su emotion (se richiesto) - CORRETTA
            if 'emotion' in required_fields:
                ref_emotion = row.get('emotion_classification', '')
                cand_emotion = json_obj.get('emotion', '')
                df.at[idx, 'em_emotion'] = 1 if exact_match_text(ref_emotion, cand_emotion) else 0
            
            # 5. EM su topic (se richiesto) - CORRETTA
            if 'topic' in required_fields:
                ref_topic = row.get('topic_classification', '')
                cand_topic = json_obj.get('topic', '')
                df.at[idx, 'em_topic'] = 1 if exact_match_text(ref_topic, cand_topic) else 0
            
            # 6. F1 su NER (se richiesto)
            if 'entities' in required_fields:
                ref_entities = row.get('entities', '')
                cand_entities = json_obj.get('entities', [])
                df.at[idx, 'f1_ner'] = calculate_ner_f1(ref_entities, cand_entities)
                
        except (json.JSONDecodeError, TypeError):
            # JSON non valido, metriche già a 0
            pass
    
    return df

print("✓ Funzione evaluate_task_file corretta per 6 task")

✓ Funzione evaluate_task_file corretta per 6 task


In [15]:
print("=" * 60)
print("CARICAMENTO E VALUTAZIONE TASK")
print("=" * 60)

# Dizionario per memorizzare i risultati
task_results = {}

for task_number in TASK_NAMES.keys():
    task_results[task_number] = evaluate_task_file(task_number)
    print("-" * 40)

print("✅ Valutazione completata per tutti i task!")

CARICAMENTO E VALUTAZIONE TASK
Valutando Task 1 (json_only)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 2 (json_translation)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 3 (json_translation_sentiment)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 4 (json_translation_sentiment_emotion)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 5 (json_translation_sentiment_emotion_topic)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 6 (json_translation_sentiment_emotion_topic_ner)...
✓ File caricato: 500 righe


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
# =============================================================================
# STEP 1: EXACT MATCH SU JSON + METRICHE TASK 1
# =============================================================================

print("=" * 60)
print("STEP 1: JSON VALIDATION + TASK 1 CONTENT FIDELITY")
print("=" * 60)

step1_results = {}

for task_number in TASK_NAMES.keys():
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Calcola metriche Step 1
    json_valid_mean = df['json_valid'].mean() * 100
    
    step1_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'total_samples': len(df)
    }
    
    # NUOVE METRICHE per Task 1
    if task_number == 1:
        # Filtra solo righe con JSON valido
        valid_json_df = df[df['json_valid'] == 1]
        
        if len(valid_json_df) > 0:
            review_em_mean = valid_json_df['review_em'].mean() * 100
            review_bleu_mean = valid_json_df['review_bleu'].mean()
            
            step1_results[task_number]['review_em'] = review_em_mean
            step1_results[task_number]['review_bleu'] = review_bleu_mean
            step1_results[task_number]['valid_json_count'] = len(valid_json_df)
            
            print(f"Task {task_number} ({task_name}):")
            print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
            print(f"  - Review Exact Match: {review_em_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
            print(f"  - Review BLEU Score: {review_bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
            print(f"  - Campioni totali: {len(df)}")
        else:
            print(f"Task {task_number} ({task_name}):")
            print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
            print(f"  - Nessun JSON valido per calcolare metriche review")
            print(f"  - Campioni totali: {len(df)}")
    else:
        print(f"Task {task_number} ({task_name}):")
        print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
        print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 1:")
for task_num, results in step1_results.items():
    if task_num == 1 and 'review_em' in results:
        print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, Review EM = {results['review_em']:.2f}%, Review BLEU = {results['review_bleu']:.2f}%")
    else:
        print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%")

In [ ]:
# =============================================================================
# STEP 2: EM JSON + BLEU TRANSLATE
# =============================================================================

print("=" * 60)
print("STEP 2: EM JSON + BLEU TRANSLATE")
print("=" * 60)

step2_results = {}

for task_number in [2, 3, 4]:  # Solo task con translate
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 2
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    
    step2_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 2:")
for task_num, results in step2_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%")


In [ ]:
# =============================================================================
# STEP 3: EM JSON + BLEU TRANSLATE + EM SENTIMENT
# =============================================================================

print("=" * 60)
print("STEP 3: EM JSON + BLEU TRANSLATE + EM SENTIMENT")
print("=" * 60)

step3_results = {}

for task_number in [3, 4]:  # Solo task con sentiment
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 3
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    
    step3_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 3:")
for task_num, results in step3_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%")


In [ ]:
# =============================================================================
# STEP 4: TUTTE LE METRICHE
# =============================================================================

print("=" * 60)
print("STEP 4: TUTTE LE METRICHE")
print("=" * 60)

step4_results = {}

for task_number in [4]:  # Solo task con NER
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 4
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    f1_ner_mean = valid_json_df['f1_ner'].mean()
    
    step4_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'f1_ner': f1_ner_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - F1 NER: {f1_ner_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 4:")
for task_num, results in step4_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, NER F1 = {results['f1_ner']:.2f}%")


In [ ]:
# =============================================================================
# ANALISI DELLA DEGRADAZIONE DELLE PERFORMANCE
# =============================================================================

print("=" * 80)
print("ANALISI DELLA DEGRADAZIONE DELLE PERFORMANCE")
print("=" * 80)

# Analisi JSON EM attraverso tutti gli step
print("📊 DEGRADAZIONE JSON EXACT MATCH:")
print("-" * 50)

json_em_progression = {}
for task_num in [1, 2, 3, 4]:
    json_values = []
    
    if task_num in step1_results:
        json_values.append(step1_results[task_num]['json_em'])
    if task_num in step2_results:
        json_values.append(step2_results[task_num]['json_em'])
    if task_num in step3_results:
        json_values.append(step3_results[task_num]['json_em'])
    if task_num in step4_results:
        json_values.append(step4_results[task_num]['json_em'])
    
    if json_values:
        json_em_progression[task_num] = json_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in json_values])}")

# NUOVA ANALISI: Review fidelity per Task 1
print("\n📊 TASK 1 - FEDELTÀ DELLA REVIEW:")
print("-" * 50)
if 1 in step1_results and 'review_em' in step1_results[1]:
    results_t1 = step1_results[1]
    print(f"Task 1 - Review Exact Match: {results_t1['review_em']:.2f}%")
    print(f"Task 1 - Review BLEU Score: {results_t1['review_bleu']:.2f}%")
else:
    print("Nessun dato disponibile per Task 1")

# Analisi BLEU attraverso gli step dove è applicabile
print("\n📊 DEGRADAZIONE BLEU TRANSLATE:")
print("-" * 50)

bleu_progression = {}
for task_num in [2, 3, 4]:
    bleu_values = []
    
    if task_num in step2_results:
        bleu_values.append(step2_results[task_num]['bleu_translate'])
    if task_num in step3_results:
        bleu_values.append(step3_results[task_num]['bleu_translate'])
    if task_num in step4_results:
        bleu_values.append(step4_results[task_num]['bleu_translate'])
    
    if bleu_values:
        bleu_progression[task_num] = bleu_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in bleu_values])}")

# Analisi Sentiment EM attraverso gli step dove è applicabile
print("\n📊 DEGRADAZIONE SENTIMENT EXACT MATCH:")
print("-" * 50)

sentiment_progression = {}
for task_num in [3, 4]:
    sentiment_values = []
    
    if task_num in step3_results:
        sentiment_values.append(step3_results[task_num]['em_sentiment'])
    if task_num in step4_results:
        sentiment_values.append(step4_results[task_num]['em_sentiment'])
    
    if sentiment_values:
        sentiment_progression[task_num] = sentiment_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in sentiment_values])}")

print("\n✅ Analisi della degradazione completata!")

In [ ]:
print("=" * 60)
print("SALVATAGGIO RISULTATI")
print("=" * 60)

# Crea un DataFrame riassuntivo migliorato
summary_data = []

# Aggiungi risultati di tutti gli step
for step_name, step_results in [
    ("Step 1", step1_results),
    ("Step 2", step2_results), 
    ("Step 3", step3_results),
    ("Step 4", step4_results)
]:
    for task_num, results in step_results.items():
        row = {
            'step': step_name,
            'task_number': task_num,
            'task_name': results['task_name'],
            'json_em': results['json_em'],
            'review_em': results.get('review_em', None),  # NUOVA COLONNA
            'review_bleu': results.get('review_bleu', None),  # NUOVA COLONNA
            'bleu_translate': results.get('bleu_translate', None),
            'em_sentiment': results.get('em_sentiment', None),
            'f1_ner': results.get('f1_ner', None),
            'total_samples': results['total_samples'],
            'valid_json_count': results.get('valid_json_count', None)
        }
        summary_data.append(row)

# Crea DataFrame riassuntivo
summary_df = pd.DataFrame(summary_data)

# Salva i risultati
output_file = f"{BASE_PATH}/evaluation_summary_enhanced_{MODEL_NAME_CLEAN}.csv"
summary_df.to_csv(output_file, index=False)

print(f"✓ Risultati salvati in: {output_file}")

# Mostra tabella riassuntiva
print("\n📋 TABELLA RIASSUNTIVA MIGLIORATA:")
print(summary_df.to_string(index=False))

# Salva anche i dataframe dettagliati per ogni task
for task_number, df in task_results.items():
    if not df.empty:
        task_name = TASK_NAMES[task_number]
        detailed_output = f"{BASE_PATH}/detailed_metrics_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}.csv"
        df.to_csv(detailed_output, index=False)
        print(f"✓ Metriche dettagliate Task {task_number} salvate in: {detailed_output}")

print("\n🎉 Script di valutazione migliorato completato!")

In [ ]:
print("\n" + "=" * 80)
print("INSIGHTS E ANALISI FINALE")
print("=" * 80)

print("🔍 INSIGHTS CHIAVE:")
print("-" * 40)

# Task 1 insights
if 1 in step1_results and 'review_em' in step1_results[1]:
    t1_results = step1_results[1]
    print(f"📌 Task 1 (Formatting Only):")
    print(f"   • JSON struttura corretta: {t1_results['json_em']:.1f}%")
    print(f"   • Fedeltà contenuto review: {t1_results['review_em']:.1f}%")
    print(f"   • Qualità trascrizione (BLEU): {t1_results['review_bleu']:.1f}%")
    
    # Calcola differenza tra validità JSON e fedeltà contenuto
    if t1_results['json_em'] > t1_results['review_em']:
        diff = t1_results['json_em'] - t1_results['review_em']
        print(f"   ⚠️  Gap formato vs contenuto: {diff:.1f}% di JSON validi hanno contenuto alterato")

# Degradazione cross-task
print(f"\n📌 Degradazione Cross-Task:")
if len([t for t in [1,2,3,4] if t in step1_results]) >= 2:
    task_complexities = []
    for t in [1,2,3,4]:
        if t in step1_results:
            task_complexities.append((t, step1_results[t]['json_em']))
    
    # Trova massimo e minimo
    max_task, max_score = max(task_complexities, key=lambda x: x[1])
    min_task, min_score = min(task_complexities, key=lambda x: x[1])
    degradation = max_score - min_score
    
    print(f"   • Migliore performance: Task {max_task} ({max_score:.1f}%)")
    print(f"   • Peggiore performance: Task {min_task} ({min_score:.1f}%)")
    print(f"   • Degradazione totale: {degradation:.1f}%")

# Translation quality degradation
if len([t for t in [2,3,4] if t in step2_results]) >= 2:
    print(f"\n📌 Degradazione Qualità Traduzione:")
    trans_scores = []
    for t in [2,3,4]:
        if t in step2_results:
            trans_scores.append((t, step2_results[t]['bleu_translate']))
    
    max_trans_task, max_trans_score = max(trans_scores, key=lambda x: x[1])
    min_trans_task, min_trans_score = min(trans_scores, key=lambda x: x[1])
    trans_degradation = max_trans_score - min_trans_score
    
    print(f"   • Migliore BLEU: Task {max_trans_task} ({max_trans_score:.1f}%)")
    print(f"   • Peggiore BLEU: Task {min_trans_task} ({min_trans_score:.1f}%)")
    print(f"   • Degradazione traduzione: {trans_degradation:.1f}%")

# Problematic areas detection
print(f"\n📌 Aree Problematiche Rilevate:")

# Controlla NER performance
if 4 in step4_results:
    ner_f1 = step4_results[4]['f1_ner']
    if ner_f1 < 10:
        print(f"   ⚠️  NER quasi fallimentare: {ner_f1:.1f}% F1-score")
        print(f"      → Possibile problema nel parsing delle entità o nel formato atteso")

# Controlla differenze significative tra task consecutivi
for i in range(1, 4):
    if i in step1_results and (i+1) in step1_results:
        curr_score = step1_results[i]['json_em']
        next_score = step1_results[i+1]['json_em']
        diff = abs(curr_score - next_score)
        if diff > 2:  # Soglia di rilevanza
            print(f"   ⚠️  Salto significativo Task {i}→{i+1}: {diff:.1f}% di differenza")

print(f"\n📊 RACCOMANDAZIONI:")
print("-" * 40)

# Task 1 recommendations
if 1 in step1_results and 'review_em' in step1_results[1]:
    t1 = step1_results[1]
    if t1['review_em'] < t1['json_em']:
        print(f"• Task 1: Migliorare fedeltà trascrizione review ({t1['review_em']:.1f}% vs {t1['json_em']:.1f}% formato)")
    elif t1['review_em'] > 95:
        print(f"• Task 1: Eccellente fedeltà contenuto ({t1['review_em']:.1f}%)")

# Translation recommendations
if step2_results:
    avg_bleu = np.mean([r['bleu_translate'] for r in step2_results.values()])
    if avg_bleu < 40:
        print(f"• Traduzione: Qualità media bassa ({avg_bleu:.1f}% BLEU) - considerare fine-tuning")
    elif avg_bleu > 50:
        print(f"• Traduzione: Qualità buona ({avg_bleu:.1f}% BLEU media)")

# NER recommendations
if 4 in step4_results and step4_results[4]['f1_ner'] < 10:
    print(f"• NER: Performance critica - verificare formato entità e algoritmo di matching")

# General stability
json_scores = [step1_results[t]['json_em'] for t in [1,2,3,4] if t in step1_results]
json_std = np.std(json_scores)
if json_std < 2:
    print(f"• Stabilità: Buona consistenza JSON across task (std: {json_std:.1f}%)")
else:
    print(f"• Stabilità: Variabilità JSON significativa across task (std: {json_std:.1f}%)")

print("\n🏁 ANALISI COMPLETA TERMINATA!")
print("=" * 80)